# 02 – VADER
Kural ve sözlük tabanlı VADER yöntemiyle İngilizce ve Türkçe test kümelerinin duygu tahmini.

- **İngilizce:** Metin doğrudan VADER'a verilir.
- **Türkçe:** VADER'ın sözlüğü İngilizce olduğu için metin önce Opus-MT ile İngilizceye çevrilir, sonra VADER uygulanır.
- **Ek deney:** Çeviri adımının katkısını görmek için Türkçe metin bir de çevrilmeden VADER'a verilir.

**Çıktılar:** `results/preds_vader_en.csv`, `results/preds_vader_tr.csv`, `results/preds_vader_tr_ham.csv`, `results/ceviri_tr_en.csv`, `results/figures/cm_vader_*.png`
**Ortam:** Google Colab, T4 GPU (çeviri için).

## 1. Kurulum ve test dosyaları
`vaderSentiment` kurulur, Drive bağlanır ve `01_veri_hazirlama` notebook'unda üretilen test kümeleri okunur.

In [1]:
!pip install -q vaderSentiment

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

KLASOR = '/content/drive/MyDrive/DuyguAnalizi'
test_en = pd.read_csv(f'{KLASOR}/data/test_en.csv')
test_tr = pd.read_csv(f'{KLASOR}/data/test_tr.csv')

print(test_en.shape, test_tr.shape)

Mounted at /content/drive
(3000, 3) (3000, 3)


## 2. VADER ve sınıf eşikleri
VADER her metin için -1 ile +1 arasında bir `compound` skoru verir. Skor, geliştiricilerin önerdiği eşiklerle üç sınıfa çevrilir:
- compound ≥ 0,05 → **Pozitif**
- compound ≤ -0,05 → **Negatif**
- arada kalanlar → **Nötr**

Altta fonksiyon üç örnek cümleyle denenir.

In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_etiket(metin):
    skor = analyzer.polarity_scores(metin)['compound']
    if skor >= 0.05:
        return 'Pozitif'
    elif skor <= -0.05:
        return 'Negatif'
    else:
        return 'Nötr'

for cumle in ['I love this airline!!', 'Worst flight ever :(', 'My flight is at 5pm.']:
    print(cumle, '→', analyzer.polarity_scores(cumle)['compound'], vader_etiket(cumle))

I love this airline!! → 0.6988 Pozitif
Worst flight ever :( → -0.7906 Negatif
My flight is at 5pm. → 0.0 Nötr


## 3. İngilizce tahmin
Test kümesindeki 3.000 tweet doğrudan VADER'a verilir. Sonuç ortak formatta (`id, text, true, pred`) `results/preds_vader_en.csv` dosyasına kaydedilir.

In [ ]:
import os

preds_en = pd.DataFrame({
    'id': test_en['id'],
    'text': test_en['text'],
    'true': test_en['label'],
    'pred': test_en['text'].apply(vader_etiket),
})

os.makedirs(f'{KLASOR}/results', exist_ok=True)
preds_en.to_csv(f'{KLASOR}/results/preds_vader_en.csv', index=False, encoding='utf-8')

print(preds_en['pred'].value_counts())
print('Doğru oranı:', (preds_en['true'] == preds_en['pred']).mean())

pred
Pozitif    1877
Negatif     578
Nötr        545
Name: count, dtype: int64
Doğru oranı: 0.558


## 4. Türkçe: çeviri modeli
Türkçe → İngilizce çeviri için `Helsinki-NLP/opus-mt-tr-en` (Opus-MT) kullanılır. transformers 5. sürümünde hazır `translation` pipeline'ı kaldırıldığı için model `AutoTokenizer` ve `AutoModelForSeq2SeqLM` ile doğrudan kullanılır. Önce 5 cümleyle çeviri kalitesine bakılır.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = 'Helsinki-NLP/opus-mt-tr-en'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to('cuda')

def cevir(cumleler):
    girdi = tokenizer(cumleler, return_tensors='pt', padding=True, truncation=True, max_length=256).to('cuda')
    with torch.no_grad():
        cikti = model.generate(**girdi, max_new_tokens=256)
    return tokenizer.batch_decode(cikti, skip_special_tokens=True)

ornekler = test_tr['text'].head(5).tolist()
for tr, en in zip(ornekler, cevir(ornekler)):
    print(tr)
    print('→', en)
    print()

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/840k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.56M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  307MB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Alman kıyı savunması da ateş açtı .
→ The German coastal defense has opened fire.

Faaliyetler gençler düzeyinde oldu .
→ The activities were at the youth level.

O zamanki yönetimlerin özel sektöre getirdiği çok büyük engeller neden olmuştur .
→ It has been a major obstacle to the private sector by the administrations at the time.

saldiriya ugradik berbat bir oteldi odalarda minibar dahi yok  havuz denilebilecek cok ufak bir yer var oyel yonetimi ve sahipleri tarafindan saldiriya ugradik   gereken yerlere sikayetlerimiz yapilmistir goruldugu anda bucak bucak kacilmasi gereken bir yer  can guvenliginiz icin asla adiminizi dahi atmayin
→ It was a terrible hotel where we attacked. There's not even a minibar in the rooms. There's a small place to call a pool.

ekonomik hızlı ve orijinal daha ne olsun :))
→ What's more economic quick and original than:)



## 5. Türkçe tahmin: çeviri + VADER
3.000 cümle 32'lik gruplar halinde çevrilir (T4 GPU'da yaklaşık 15 dakika), ardından çevirilere VADER uygulanır. Tahmin dosyasında (`results/preds_vader_tr.csv`) orijinal Türkçe metin tutulur; çeviriler ayrıca `results/ceviri_tr_en.csv` dosyasına kaydedilir.

In [ ]:
from tqdm.auto import tqdm

cumleler = test_tr['text'].tolist()
ceviriler = []
for i in tqdm(range(0, len(cumleler), 32)):
    ceviriler += cevir(cumleler[i:i+32])

preds_tr = pd.DataFrame({
    'id': test_tr['id'],
    'text': test_tr['text'],
    'true': test_tr['label'],
    'pred': [vader_etiket(c) for c in ceviriler],
})
preds_tr.to_csv(f'{KLASOR}/results/preds_vader_tr.csv', index=False, encoding='utf-8')

pd.DataFrame({'id': test_tr['id'], 'text': test_tr['text'], 'ceviri': ceviriler}).to_csv(
    f'{KLASOR}/results/ceviri_tr_en.csv', index=False, encoding='utf-8')

print(preds_tr['pred'].value_counts())
print('Doğru oranı:', (preds_tr['true'] == preds_tr['pred']).mean())

  0%|          | 0/94 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

pred
Pozitif    1360
Nötr        883
Negatif     757
Name: count, dtype: int64
Doğru oranı: 0.6786666666666666


### 5.1 Kontrol
Çeviri hücresinin sonucu uyarı mesajları arasında kaldığı için tahmin dağılımı ve doğruluk oranı burada tekrar yazdırılır. Son satır, çevrilen cümle sayısını kontrol eder; 3.000 çıkması her cümlenin çevrildiğini gösterir.

In [ ]:
print(preds_tr['pred'].value_counts())
print('Doğru oranı:', (preds_tr['true'] == preds_tr['pred']).mean())
print(len(ceviriler))

pred
Pozitif    1360
Nötr        883
Negatif     757
Name: count, dtype: int64
Doğru oranı: 0.6786666666666666
3000


## 6. Değerlendirme
İki tahmin dosyası ortak `src/evaluate.py` fonksiyonuyla ölçülür: accuracy, macro-F1, sınıf başına hata sayısı ve confusion matrix görseli (`results/figures/cm_vader_en.png`, `cm_vader_tr.png`).

In [3]:
import sys
sys.path.append(f'{KLASOR}/src')
from evaluate import evaluate

for dil in ['en', 'tr']:
    sonuc = evaluate(f'{KLASOR}/results/preds_vader_{dil}.csv', gorsel_klasoru=f'{KLASOR}/results/figures')
    print(f'--- VADER {dil.upper()} ---')
    for anahtar, deger in sonuc.items():
        print(f'{anahtar}: {deger}')
    print()

--- VADER EN ---
dosya: preds_vader_en.csv
n: 3000
accuracy: 0.558
macro_f1: 0.5368
hatalar: {'Pozitif': 91, 'Negatif': 567, 'Nötr': 668}
gecersiz: 0
gorsel: /content/drive/MyDrive/DuyguAnalizi/results/figures/cm_vader_en.png

--- VADER TR ---
dosya: preds_vader_tr.csv
n: 3000
accuracy: 0.6787
macro_f1: 0.6759
hatalar: {'Pozitif': 178, 'Negatif': 455, 'Nötr': 331}
gecersiz: 0
gorsel: /content/drive/MyDrive/DuyguAnalizi/results/figures/cm_vader_tr.png



## 7. Ek deney: ham Türkçe VADER (çevirisiz)
Çeviri adımının katkısını ölçmek için Türkçe metin çevrilmeden doğrudan VADER'a verilir. VADER Türkçe kelimeleri tanımadığı için tahminlerin büyük çoğunluğu Nötr çıkar (accuracy ≈ 0,36). Çeviri ile accuracy ≈ 0,68'e yükselir. Sonuç `results/preds_vader_tr_ham.csv` ve `cm_vader_tr_ham.png` olarak kaydedilir.

In [4]:
preds_tr_ham = pd.DataFrame({
    'id': test_tr['id'],
    'text': test_tr['text'],
    'true': test_tr['label'],
    'pred': test_tr['text'].apply(vader_etiket),
})
preds_tr_ham.to_csv(f'{KLASOR}/results/preds_vader_tr_ham.csv', index=False, encoding='utf-8')

print(preds_tr_ham['pred'].value_counts())
print()

sonuc = evaluate(f'{KLASOR}/results/preds_vader_tr_ham.csv', gorsel_klasoru=f'{KLASOR}/results/figures')
print('--- VADER TR (ham, çevirisiz) ---')
for anahtar, deger in sonuc.items():
    print(f'{anahtar}: {deger}')

pred
Nötr       2837
Pozitif     120
Negatif      43
Name: count, dtype: int64

--- VADER TR (ham, çevirisiz) ---
dosya: preds_vader_tr_ham.csv
n: 3000
accuracy: 0.364
macro_f1: 0.2324
hatalar: {'Pozitif': 932, 'Negatif': 970, 'Nötr': 6}
gecersiz: 0
gorsel: /content/drive/MyDrive/DuyguAnalizi/results/figures/cm_vader_tr_ham.png
